# Strategic Drift — Hoberg-Phillips (2016/2023) with FinBERT Embeddings

Adapts the text-based industry classification framework of Hoberg & Phillips to
transformer embeddings. Five stages:

| Stage | Description | Output |
|---|---|---|
| 1 | Text already preprocessed | `text_chunks.parquet` |
| 2 | FinBERT embedding per firm-year | `embeddings_panel.parquet` |
| 3 | K-means (K=25) on 2010 embeddings | centroids array |
| 4 | Firm-to-industry cosine scoring | similarity profiles |
| 5 | Strategic drift measures | `strategic_drift.parquet` |

**Model**: `yiyanghkust/finbert-pretrain` — pre-trained on 4.9B tokens of financial text,
**not** fine-tuned for sentiment. Embeddings capture general financial semantic content.

**Requires** GPU runtime (`Runtime → Change runtime type → T4 GPU`).

## 0. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

## 1. Configuration

In [ ]:
import os

CONFIG = {
    'chunks_path':         '/content/drive/MyDrive/FML_project_4/text_chunks.parquet',
    'usable_pairs_path':   '/content/drive/MyDrive/FML_project_4/usable_pairs.parquet',
    'analysis_panel_path': '/content/drive/MyDrive/FML_project_4/analysis_panel.parquet',
    'output_folder':       '/content/drive/MyDrive/FML_project_4',

    # FinBERT pre-trained (NOT the sentiment fine-tuned version)
    # Captures general financial semantic content, not tone
    'embed_model': 'yiyanghkust/finbert-pretrain',

    # Sections combined for firm strategic identity (Items 1 + 7)
    'sections': ['item_1', 'item_7'],

    # K-means
    'K':         25,      # clusters; ~20 firms/cluster for ~500 firms/year
    'base_year': 2010,    # mid-sample; avoids 2007-2009 crisis distortion
    'K_range':   [20, 25, 30],  # robustness check

    # Inference
    'embed_batch': 16,    # chunks per GPU forward pass; reduce to 8 if OOM
    'cik_batch':   50,    # CIKs per checkpoint file
}
print('Config OK')

## 2. Install dependencies

In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q',
    'transformers', 'torch', 'pyarrow', 'scikit-learn', 'tqdm'], check=True)
print('Dependencies ready')

## 3. Load data

In [ ]:
import pandas as pd
import numpy as np
import torch
import gc
import glob as _glob
from tqdm.auto import tqdm
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cpu':
    print('WARNING: No GPU. Embedding stage will take many hours.')

usable = pd.read_parquet(CONFIG['usable_pairs_path'])
usable['cik']  = usable['cik'].astype(str).str.lstrip('0')
usable['year'] = usable['year'].astype('Int64')
usable_set     = set(zip(usable['cik'], usable['year'].astype(int)))
print(f'Usable pairs: {len(usable_set):,}')

# Load item_1 + item_7 chunks only
print('Loading chunks ...')
chunks = pd.read_parquet(CONFIG['chunks_path'])
chunks['cik']  = chunks['cik'].astype(str).str.lstrip('0')
chunks['year'] = chunks['year'].astype('Int64')
chunks = chunks[chunks['section'].isin(CONFIG['sections'])]
chunks = chunks[chunks.apply(lambda r: (r['cik'], int(r['year'])) in usable_set, axis=1)]
print(f'Chunks: {len(chunks):,}  |  CIKs: {chunks["cik"].nunique():,}')

ALL_CIKS = sorted(chunks['cik'].unique().tolist())

## 4. Stage 2 — FinBERT Embeddings

For each chunk: extract last hidden-layer token representations, apply mean pooling
over non-padding tokens → 768-dim vector. Then average all chunk vectors for the
same (cik, year) → firm-year embedding **e**_{i,t} ∈ ℝ⁷⁶⁸.

**Resume**: checkpoints every 50 CIKs to `/content/temp_drift/`.

In [ ]:
EMB_PATH  = os.path.join(CONFIG['output_folder'], 'embeddings_panel.parquet')
TEMP_DIR  = '/content/temp_drift'
os.makedirs(TEMP_DIR, exist_ok=True)

if os.path.exists(EMB_PATH):
    print('embeddings_panel.parquet exists — skipping Stage 2.')
    emb_panel = pd.read_parquet(EMB_PATH)
else:
    from transformers import AutoTokenizer, AutoModel

    tokenizer = AutoTokenizer.from_pretrained(CONFIG['embed_model'])
    model     = AutoModel.from_pretrained(CONFIG['embed_model'])
    model.eval().to(DEVICE)
    print(f'Model loaded: {CONFIG["embed_model"]}')

    def embed_texts(texts: list[str]) -> np.ndarray:
        """Mean-pool last hidden state over non-padding tokens → (n, 768)."""
        enc = tokenizer(texts, padding=True, truncation=True,
                        max_length=512, return_tensors='pt')
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        with torch.no_grad():
            hidden = model(**enc).last_hidden_state   # (B, seq, 768)
        mask    = enc['attention_mask'].unsqueeze(-1).float()
        summed  = (hidden * mask).sum(dim=1)           # (B, 768)
        counts  = mask.sum(dim=1).clamp(min=1e-9)      # (B, 1)
        return (summed / counts).cpu().float().numpy()

    cik_batches = [ALL_CIKS[i:i+CONFIG['cik_batch']]
                   for i in range(0, len(ALL_CIKS), CONFIG['cik_batch'])]
    done = {int(os.path.basename(f)[4:8])
            for f in _glob.glob(f'{TEMP_DIR}/emb_*.parquet')}
    print(f'{len(cik_batches)} batches — {len(done)} done — {len(cik_batches)-len(done)} remaining')

    for b_num, batch_ciks in enumerate(tqdm(cik_batches, desc='Embedding')):
        tmp = f'{TEMP_DIR}/emb_{b_num:04d}.parquet'
        if os.path.exists(tmp):
            continue

        sub   = chunks[chunks['cik'].isin(batch_ciks)]
        texts = sub['text'].tolist()
        B     = CONFIG['embed_batch']
        embs  = np.vstack([embed_texts(texts[i:i+B]) for i in range(0, len(texts), B)])

        result = sub[['cik','year']].copy().reset_index(drop=True)
        result['emb'] = list(embs)

        # Average chunk embeddings → firm-year embedding
        agg_rows: list[dict] = []
        for (cik, year), grp in result.groupby(['cik','year']):
            mean_emb = np.stack(grp['emb'].values).mean(axis=0)
            agg_rows.append({'cik': cik, 'year': year,
                             'emb': mean_emb.tolist()})
        pd.DataFrame(agg_rows).to_parquet(tmp, index=False, compression='snappy')
        del sub, texts, embs, result, agg_rows; gc.collect()

    # Consolidate
    files = sorted(f for f in _glob.glob(f'{TEMP_DIR}/emb_*.parquet')
                   if os.path.getsize(f) > 0)
    emb_panel = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
    emb_panel['year'] = emb_panel['year'].astype('Int64')
    emb_panel.to_parquet(EMB_PATH, index=False, compression='snappy')
    print(f'Saved embeddings_panel.parquet  {len(emb_panel):,} firm-years')

print(f'Embedding panel: {len(emb_panel):,} rows, embedding dim: {len(emb_panel["emb"].iloc[0])}')
emb_panel.head(3)

## 5. Stage 3 — K-Means Industry Clustering (base year 2010)

Cluster S&P 500 firms using their 2010 embeddings. Centroids are **fixed** for all years
(Hoberg & Phillips 2023). Robustness checks with K ∈ {20, 25, 30}.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

# Build embedding matrix for base year
base = emb_panel[emb_panel['year'] == CONFIG['base_year']].copy()
X_base = np.vstack(base['emb'].tolist())   # (N_firms, 768)
print(f'Base year {CONFIG["base_year"]}: {len(base)} firms, embedding matrix {X_base.shape}')

# ── Robustness: elbow + silhouette for K in K_range ──
inertias:    list[float] = []
silhouettes: list[float] = []

for K in CONFIG['K_range']:
    km  = KMeans(n_clusters=K, random_state=42, n_init=10)
    lbl = km.fit_predict(X_base)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_base, lbl, sample_size=min(500, len(base)))
    silhouettes.append(sil)
    print(f'  K={K:3d}  inertia={km.inertia_:.1f}  silhouette={sil:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(CONFIG['K_range'], inertias, marker='o', color='#1f77b4')
axes[0].set_title('Elbow method')
axes[0].set_xlabel('K'); axes[0].set_ylabel('Inertia')
axes[0].grid(linestyle='--', alpha=0.4)
axes[1].plot(CONFIG['K_range'], silhouettes, marker='o', color='#2ca02c')
axes[1].set_title('Silhouette score')
axes[1].set_xlabel('K'); axes[1].set_ylabel('Score')
axes[1].grid(linestyle='--', alpha=0.4)
plt.suptitle(f'K-means diagnostics on {CONFIG["base_year"]} embeddings', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_folder'], 'kmeans_diagnostics.png'),
            dpi=150, bbox_inches='tight')
plt.show()

# ── Fit final model with chosen K ──
K = CONFIG['K']
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
base_labels = kmeans.fit_predict(X_base)
CENTROIDS = kmeans.cluster_centers_   # (K, 768) — fixed for all years

# Map CIK → base-year industry assignment k*
base_assignment: dict[str, int] = dict(zip(base['cik'].tolist(), base_labels.tolist()))
print(f'\nK={K} fitted. Cluster sizes:')
unique, counts = np.unique(base_labels, return_counts=True)
for k, c in sorted(zip(unique, counts)):
    print(f'  Cluster {k:2d}: {c} firms')

## 6. Stage 4 — Firm-to-Industry Cosine Similarity Profiles

For each (cik, year), compute cosine similarity to all K centroids → K-element profile.
Each firm's primary industry `k*` is fixed at its 2010 assignment.

In [ ]:
E = np.vstack(emb_panel['emb'].tolist())          # (N, 768)
S = cos_sim(E, CENTROIDS)                         # (N, K) — all cosine similarities

sim_cols = [f'sim_k{k}' for k in range(K)]
sim_df   = pd.DataFrame(S, columns=sim_cols)
sim_df   = pd.concat([emb_panel[['cik','year']].reset_index(drop=True), sim_df], axis=1)

# Nearest centroid each year (may change over time)
sim_df['nearest_k'] = S.argmax(axis=1)

# Primary industry = base-year assignment (fixed, from Stage 3)
sim_df['k_star'] = sim_df['cik'].map(base_assignment)

print(f'Similarity profiles: {sim_df.shape}')
print(f'CIKs with base-year assignment: {sim_df["k_star"].notna().sum():,} / {len(sim_df):,}')
sim_df.head(3)

## 7. Stage 5 — Strategic Drift Measures

Three measures per firm-year:

| Measure | Formula | Interpretation |
|---|---|---|
| `drift` | `1 − cos(e_{i,t}, μ_{k*_i})` | Distance from original industry centroid |
| `delta_drift` | `drift_t − drift_{t−1}` | Speed of strategic repositioning |
| `self_sim` | `cos(e_{i,t}, e_{i,t−1})` | YoY self-similarity (Cohen, Malloy & Nguyen 2020) |
| `migration` | `1` if `nearest_k_t ≠ nearest_k_{t−1}` | Discrete industry boundary crossing |

In [ ]:
DRIFT_PATH = os.path.join(CONFIG['output_folder'], 'strategic_drift.parquet')

# Drift = 1 − similarity to base-year centroid
def get_centroid_sim(row: pd.Series) -> float:
    k = row['k_star']
    if pd.isna(k):
        return np.nan
    return row[f'sim_k{int(k)}']

sim_df['centroid_sim'] = sim_df.apply(get_centroid_sim, axis=1)
sim_df['drift']        = 1.0 - sim_df['centroid_sim']

# Sort and compute time-series measures per CIK
sim_df = sim_df.sort_values(['cik','year'])
sim_df['delta_drift'] = sim_df.groupby('cik')['drift'].diff()
sim_df['migration']   = (sim_df.groupby('cik')['nearest_k']
                                .transform(lambda x: x.diff().abs().clip(upper=1))
                                .fillna(0).astype(int))

# YoY self-similarity using raw embeddings
self_sim_rows: list[dict] = []
for cik, grp in emb_panel.groupby('cik'):
    grp = grp.sort_values('year').reset_index(drop=True)
    emb_mat = np.vstack(grp['emb'].tolist())
    for i in range(1, len(grp)):
        sim = float(cos_sim([emb_mat[i]], [emb_mat[i-1]])[0, 0])
        self_sim_rows.append({'cik': cik, 'year': int(grp['year'].iloc[i]),
                              'self_sim': sim, 'self_change': 1.0 - sim})

self_sim_df = pd.DataFrame(self_sim_rows)
self_sim_df['year'] = self_sim_df['year'].astype('Int64')

# Merge
keep = ['cik','year','drift','delta_drift','centroid_sim','nearest_k','k_star','migration']
drift_df = sim_df[keep].merge(self_sim_df, on=['cik','year'], how='left')

# Drop pre-base-year rows: centroids were estimated on base_year data, so applying them
# to pre-base_year firms uses future information (look-ahead bias).
# Only keep years >= base_year for all downstream analysis.
drift_df = drift_df[drift_df['year'] >= CONFIG['base_year']].copy()

drift_df.to_parquet(DRIFT_PATH, index=False, compression='snappy')

print(f'Saved strategic_drift.parquet  {len(drift_df):,} rows')
print(f'Year range: {int(drift_df["year"].min())} – {int(drift_df["year"].max())}')
print(drift_df[['drift','delta_drift','self_sim','migration']].describe().round(4).to_string())
drift_df.head(4)

## 8. Drift visualisations

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Mean drift per year
annual_drift = drift_df.groupby('year')['drift'].mean().reset_index()
axes[0].plot(annual_drift['year'], annual_drift['drift'],
             marker='o', color='#1f77b4', linewidth=1.5)
axes[0].set_title('Mean strategic drift from 2010 centroid')
axes[0].set_xlabel('Year'); axes[0].set_ylabel('Drift (1 − cosine sim)')
axes[0].axvline(CONFIG['base_year'], color='red', linestyle='--', alpha=0.6, label='Base year')
axes[0].legend(); axes[0].grid(linestyle='--', alpha=0.4)

# Industry migration rate per year
mig_rate = drift_df.groupby('year')['migration'].mean() * 100
axes[1].bar(mig_rate.index, mig_rate.values, color='#ff7f0e', alpha=0.8)
axes[1].set_title('Industry migration rate (% firms crossing boundary)')
axes[1].set_xlabel('Year'); axes[1].set_ylabel('% firms migrated')
axes[1].grid(axis='y', linestyle='--', alpha=0.4)

# Distribution of drift values (pooled)
drift_vals = drift_df['drift'].dropna()
axes[2].hist(drift_vals, bins=40, color='#2ca02c', edgecolor='white', alpha=0.85)
axes[2].axvline(drift_vals.median(), color='red', linestyle='--',
                label=f'Median {drift_vals.median():.3f}')
axes[2].set_title('Pooled distribution of strategic drift')
axes[2].set_xlabel('Drift'); axes[2].set_ylabel('# Observations')
axes[2].legend(); axes[2].grid(axis='y', linestyle='--', alpha=0.4)

plt.suptitle('Strategic Drift — FinBERT + K-Means (2010 base)', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_folder'], 'strategic_drift_summary.png'),
            dpi=150, bbox_inches='tight')
plt.show()

## 9. Sorting diagnostic

Quintile-sort on each drift signal per year, compute mean buy-and-hold return over
the 12-month forward window attached to each filing.

**This is a diagnostic only** — it shows whether drift correlates with subsequent returns.
It is NOT a portfolio backtest:
- No transaction costs, rebalancing, or position sizing
- No risk-factor adjustment (market, size, value, momentum)
- Standard errors are not Newey-West corrected

Note: pre-2010 rows were dropped in Stage 5, so this diagnostic covers 2010 onwards.
A proper test requires Fama-MacBeth regressions with controls.

In [ ]:
panel = pd.read_parquet(CONFIG['analysis_panel_path'])
panel['cik']  = panel['cik'].astype(str).str.lstrip('0')
panel['year'] = panel['year'].astype('Int64')
panel = panel.merge(drift_df[['cik','year','drift','delta_drift','self_change']],
                    on=['cik','year'], how='left')

bahr = (
    panel.groupby(['cik','year'])
         .apply(lambda g: (1 + g['monthly_return'].fillna(0)).prod() - 1)
         .reset_index(name='annual_return')
)

DRIFT_SIGNALS = {
    'drift':        'Drift from 2010 centroid',
    'delta_drift':  'ΔDrift (speed of repositioning)',
    'self_change':  'Self-similarity change',
}

results: list[dict] = []
for sig, label in DRIFT_SIGNALS.items():
    if sig not in panel.columns:
        continue
    sig_df = (
        panel.drop_duplicates(['cik','year'])[['cik','year', sig]]
             .merge(bahr, on=['cik','year'])
             .dropna(subset=[sig, 'annual_return'])
    )
    sig_df['q'] = sig_df.groupby('year')[sig].transform(
        lambda x: pd.qcut(x, 5, labels=False, duplicates='drop'))
    q_ret = sig_df.groupby('q')['annual_return'].mean()
    ls    = (q_ret.get(4, np.nan) - q_ret.get(0, np.nan)) * 100
    results.append({'Signal': label, 'L/S (%)': round(ls, 2),
                    'Q1 (%)': round(q_ret.get(0, np.nan)*100, 2),
                    'Q5 (%)': round(q_ret.get(4, np.nan)*100, 2)})

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#2ca02c' if v >= 0 else '#d62728' for v in results_df['L/S (%)']]
ax.bar(results_df['Signal'], results_df['L/S (%)'], color=colors, alpha=0.85, edgecolor='white')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Strategic Drift — Long/Short Q5−Q1 Annual Return', fontsize=12)
ax.set_ylabel('Mean Annual Return (%)')
ax.tick_params(axis='x', rotation=10)
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_folder'], 'drift_portfolio_sort.png'),
            dpi=150, bbox_inches='tight')
plt.show()